# 🌊 Pacific Sea Level Variability Analysis
**Satellite Altimetry × Tide Gauge Cross-Validation | 1993–2018**

This notebook analyzes 25 years of Pacific Ocean sea level data by fusing satellite altimetry (gridded, ~19,400 grid points) with in-situ tide gauge records from Honolulu, San Diego, and Mera. The analysis decomposes sea level into long-term trends, seasonal cycles, and residual anomalies, then maps how well each tide gauge predicts basin-wide sea level variability.

---
**Data sources:**
- `ALT_Pacific.mat` — AVISO satellite altimetry SLA, 121 lat × 161 lon × 311 months
- `TGdata.mat` — UHSLC tide gauge records, 3 stations × 311 months

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sb
import scipy.io
from scipy import stats
from sklearn import linear_model
from mpl_toolkits.axes_grid1 import make_axes_locatable
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

## 2. Load Data

In [ ]:
# --- Altimeter data ---
data = scipy.io.loadmat('ALT_Pacific.mat')
sl  = data['ALT']['sl'][0][0]          # shape: (311, 121, 161) — time × lat × lon
t   = data['ALT']['t'][0][0].flatten() # shape: (311,) — datenum, monthly
lon = data['ALT']['lon'][0][0]         # shape: (161, 1)
lat = data['ALT']['lat'][0][0][0].reshape(121, 1)  # shape: (121, 1)

print(f'Altimeter: {sl.shape[0]} months | {len(lat)} lats | {len(lon)} lons')

# --- Tide gauge data ---
tgdata = scipy.io.loadmat('TGdata.mat')
tgt   = tgdata['TG']['t'][0][0].flatten()    # (311,)
tgsl  = tgdata['TG']['sl'][0][0]             # (311, 3) — Honolulu, San Diego, Mera
tglat = tgdata['TG']['lat'][0][0][0]         # (3,)
tglon = tgdata['TG']['lon'][0][0][0]         # (3,)

print(f'Tide gauges: Honolulu ({tglat[0]:.1f}°N), San Diego ({tglat[1]:.1f}°N), Mera ({tglat[2]:.1f}°N)')

In [ ]:
# Convert MATLAB datenums to pandas Timestamps
MATLAB_EPOCH = 719529
alt_timestamps = pd.to_datetime(t   - MATLAB_EPOCH, unit='D')
tg_timestamps  = pd.to_datetime(tgt - MATLAB_EPOCH, unit='D')

## 3. Sea Level Range
Peak-to-peak range at each grid point — a first-pass look at where sea level variability is largest.

In [ ]:
sl_range = np.full((len(lat), len(lon)), np.nan)

for i in range(len(lon)):
    for j in range(len(lat)):
        sl_temp = np.squeeze(sl[:, j, i])
        if not np.isnan(np.mean(sl_temp)):
            sl_range[j][i] = sl_temp.max() - sl_temp.min()

fig, ax = plt.subplots(figsize=(9, 5))
img = ax.imshow(sl_range, origin='lower',
                extent=[lon[0][0], lon[-1][0], lat[0][0], lat[-1][0]])
ax.set(xlabel='Longitude', ylabel='Latitude', title='Sea Level Range (1993–2018)')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='3%', pad=0.4)
fig.colorbar(img, cax=cax).set_label('metres')
plt.savefig('../outputs/figures/sl_range.png')
plt.show()

## 4. Trend & Seasonal Cycle Decomposition

At every grid point, fit:

$$SL(t) = \underbrace{\beta_0 t}_{\text{trend}} + \underbrace{\mu}_{\text{mean}} + \underbrace{A_1\cos\omega t + B_1\sin\omega t + A_2\cos 2\omega t + B_2\sin 2\omega t}_{\text{seasonal cycle}} + \varepsilon$$

where $\omega = 2\pi / 365.25$ and $\varepsilon$ is the sea level anomaly (SLA).

In [ ]:
# Pre-compute harmonic terms
arg     = 2 * math.pi * t / 365.25
cosarg  = np.cos(arg)
sinarg  = np.sin(arg)
cos2arg = np.cos(2 * arg)
sin2arg = np.sin(2 * arg)

# Design matrix: [trend, mean, cos(ω), sin(ω), cos(2ω), sin(2ω)]
X = np.column_stack((t, np.ones(len(t)), cosarg, sinarg, cos2arg, sin2arg))

regr = linear_model.LinearRegression(fit_intercept=False)

sl_trend      = np.full((len(lat), len(lon)), np.nan)
sl_season_std = np.full((len(lat), len(lon)), np.nan)
sl_tms        = np.full((len(lat), len(lon), 6), np.nan)

# Design matrix centered on mean(t) for stable coefficient extraction
X_centered = np.column_stack((t - np.mean(t), np.ones(len(t)),
                               cosarg, sinarg, cos2arg, sin2arg))

for i in range(len(lon)):
    for j in range(len(lat)):
        sl_temp = np.squeeze(sl[:, j, i])
        if not np.isnan(np.mean(sl_temp)):
            regr.fit(X, sl_temp)
            # Trend in mm/year
            sl_trend[j][i] = regr.coef_[0] * 1000 * 365.25
            # Seasonal cycle std
            c = regr.coef_
            sl_season_std[j][i] = (1 / math.sqrt(2)) * math.sqrt(
                c[2]**2 + c[3]**2 + c[4]**2 + c[5]**2)
            # Store all coefficients (centered)
            regr.fit(X_centered, sl_temp)
            sl_tms[j][i] = regr.coef_

print('Coefficient array shape:', sl_tms.shape)

In [ ]:
def plot_map(data, title, cbar_label, fname, **imshow_kwargs):
    fig, ax = plt.subplots(figsize=(9, 5))
    img = ax.imshow(data, origin='lower',
                    extent=[lon[0][0], lon[-1][0], lat[0][0], lat[-1][0]],
                    **imshow_kwargs)
    ax.set(xlabel='Longitude', ylabel='Latitude', title=title)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='3%', pad=0.4)
    fig.colorbar(img, cax=cax).set_label(cbar_label)
    plt.savefig(f'../outputs/figures/{fname}')
    plt.show()

plot_map(sl_trend,      'Sea Level Trend (1993–2018)',              'mm/year', 'sl_trend.png')
plot_map(sl_season_std, 'Std. Deviation of Seasonal Cycle',         'm',       'sl_seasonal_std.png')

## 5. Tide Gauge Trend Fits
Apply the same decomposition to each tide gauge station.

In [ ]:
sl_ho, sl_sd, sl_me = tgsl[:, 0], tgsl[:, 1], tgsl[:, 2]

arg_tg     = 2 * math.pi * tgt / 365.25
X_tg = np.column_stack((
    tgt - np.mean(tgt), np.ones(len(tgt)),
    np.cos(arg_tg), np.sin(arg_tg),
    np.cos(2 * arg_tg), np.sin(2 * arg_tg)
))

regr = linear_model.LinearRegression(fit_intercept=False)

results = {}
for name, sl_tg in [('Honolulu', sl_ho), ('San Diego', sl_sd), ('Mera', sl_me)]:
    regr.fit(X_tg, sl_tg)
    trend = regr.coef_[0] * 365.25 * 1000
    results[name] = {'coeffs': regr.coef_.copy(), 'trend_mm_yr': trend}
    print(f'{name:12s}  trend: {trend:.2f} mm/year')

In [ ]:
# Plot San Diego with fit overlay
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(tg_timestamps, sl_sd, lw=0.8, label='Observed')
ax.plot(tg_timestamps, np.dot(X_tg, results['San Diego']['coeffs']),
        lw=1.5, alpha=0.8, label='Trend + Seasonal fit')
ax.set(xlabel='Year', ylabel='Metres', title='San Diego Sea Level (1993–2018)')
ax.legend()
plt.savefig('../outputs/figures/san_diego_fit.png')
plt.show()

## 6. Sea Level Anomalies
Subtract the fitted trend + seasonal cycle to isolate the residual anomaly signal.

In [ ]:
# Tide gauge anomalies
sl_ho_anom = sl_ho - np.dot(X_tg, results['Honolulu']['coeffs'])
sl_sd_anom = sl_sd - np.dot(X_tg, results['San Diego']['coeffs'])
sl_me_anom = sl_me - np.dot(X_tg, results['Mera']['coeffs'])

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
plt.subplots_adjust(hspace=0.4)
for ax, anom, name in zip(axes,
                           [sl_ho_anom, sl_sd_anom, sl_me_anom],
                           ['Honolulu', 'San Diego', 'Mera']):
    ax.plot(tg_timestamps, anom, lw=0.8)
    ax.set(title=f'{name} Sea Level Anomalies', xlabel='Year', ylabel='Metres')
    ax.axhline(0, color='k', lw=0.5, ls='--')
plt.savefig('../outputs/figures/tg_anomalies.png')
plt.show()

In [ ]:
# Altimeter anomalies — subtract fitted model at each grid point
X_alt_centered = np.column_stack((
    t - np.mean(t), np.ones(len(t)),
    cosarg, sinarg, cos2arg, sin2arg
))

sl_anom = np.full((len(lat), len(lon), 311), np.nan)

for i in range(len(lon)):
    for j in range(len(lat)):
        if not np.isnan(sl_tms[j, i, 0]):
            sl_anom[j, i, :] = sl[:, j, i] - np.dot(X_alt_centered, sl_tms[j, i])

print('Anomaly array shape:', sl_anom.shape)

## 7. Correlation: Tide Gauge ↔ Altimeter Grid

Pearson correlation between each tide gauge anomaly time series and the altimeter anomaly at every ocean grid point. High correlation indicates the gauge signal is "felt" across that region — key for understanding ENSO teleconnections.

In [ ]:
tg_ho_corr = np.full((len(lat), len(lon)), np.nan)
tg_sd_corr = np.full((len(lat), len(lon)), np.nan)
tg_me_corr = np.full((len(lat), len(lon)), np.nan)

for i in range(len(lon)):
    for j in range(len(lat)):
        tg_ho_corr[j][i] = np.corrcoef(sl_ho_anom, sl_anom[j, i, :])[0][1]
        tg_sd_corr[j][i] = np.corrcoef(sl_sd_anom, sl_anom[j, i, :])[0][1]
        tg_me_corr[j][i] = np.corrcoef(sl_me_anom, sl_anom[j, i, :])[0][1]

for corr, name in [(tg_ho_corr, 'Honolulu'), (tg_sd_corr, 'San Diego'), (tg_me_corr, 'Mera')]:
    plot_map(corr, f'Correlation — {name} Tide Gauge vs. Altimeter',
             'Pearson r', f'corr_{name.lower().replace(" ", "_")}.png')

## 8. Regression Coefficients
The regression coefficient (gain) shows *by how much* the tide gauge anomaly changes for a 1 m change in altimeter SLA at each point.

In [ ]:
tg_ho_regr = np.full((len(lat), len(lon)), np.nan)
tg_sd_regr = np.full((len(lat), len(lon)), np.nan)
tg_me_regr = np.full((len(lat), len(lon)), np.nan)

regr = linear_model.LinearRegression(fit_intercept=False)

for i in range(len(lon)):
    for j in range(len(lat)):
        sl_temp = sl_anom[j, i, :].reshape(311, 1)
        if not np.isnan(np.mean(sl_temp)):
            regr.fit(sl_temp, sl_ho_anom.reshape(311, 1))
            tg_ho_regr[j][i] = regr.coef_[0]
            regr.fit(sl_temp, sl_sd_anom.reshape(311, 1))
            tg_sd_regr[j][i] = regr.coef_[0]
            regr.fit(sl_temp, sl_me_anom.reshape(311, 1))
            tg_me_regr[j][i] = regr.coef_[0]

# Clip colormap to [-0.4, 0.6] to reduce outlier influence
for regr_map, name in [(tg_ho_regr, 'Honolulu'), (tg_sd_regr, 'San Diego'), (tg_me_regr, 'Mera')]:
    plot_map(regr_map, f'Regression Coefficients — {name}',
             'Gain (m/m)', f'regr_{name.lower().replace(" ", "_")}.png',
             vmin=-0.4, vmax=0.6)

## 9. Summary & Interpretation

### Trend Results
| Station | Trend |
|---|---|
| Honolulu | +2.02 mm/year |
| San Diego | +4.08 mm/year |
| Mera | +4.98 mm/year |

All three stations show positive sea level rise over the 1993–2018 period, consistent with global mean sea level rise estimates from satellite altimetry (~3–4 mm/year globally).

### Spatial Patterns
**San Diego** exhibits the most distinctive spatial correlation structure — positive along the North and South American coasts and across the equatorial belt, negative in the western tropical Pacific (Papua New Guinea, Australia). This dipole pattern is a hallmark of **ENSO**: during El Niño, sea level rises in the eastern Pacific and falls in the west; the reverse occurs during La Niña.

**Honolulu** sits near the node of the ENSO pattern, yielding more moderate but broadly positive correlations across the central Pacific.

**Mera** reflects North Pacific dynamics and western boundary current variability, with its highest correlations concentrated along the Asian coast — and importantly, opposing correlation sign north of Hawaii compared to San Diego and Honolulu, highlighting fundamentally different drivers at that location.

### Climate Relevance
These findings underscore that regional sea level variability is not simply a uniform signal — ENSO-driven redistribution can temporarily mask or amplify underlying long-term trends. For coastal risk assessment and climate adaptation planning, understanding these teleconnections is as important as the mean trend itself.